In [ ]:
from pathlib import Path
import gc
import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle

CLONE_COLORS = {
    'A': '#FFB6B9', 'B': '#6096B4', 'C': '#FFD758', 'D': '#FF9100',
    'E': '#238b45', 'F': '#B648F1', 'G': '#B17648',
}
OTHER_CLONE_COLOR = '#D9D9D9'
MSEP_COLORS = {
    'M-SEP1': '#e41a1c', 'M-SEP2': '#5b8dc2', 'M-SEP3': '#FCAD38',
    'M-SEP4': '#b3de69', 'M-SEP5': '#A290B7',
}
def clean_labels(values):
    s = pd.Series(values, dtype='string').str.strip()
    return s.mask(s.str.lower().isin(['', 'nan', 'none', 'na', '<na>', 'unassigned', 'conflict']))

def canonical_hex_ids(values, sample_id):
    s = clean_labels(values)
    prefix = sample_id + '__'
    return s.where(s.isna() | s.str.startswith(prefix, na=False), prefix + s)

def accepted_flag(values):
    s = pd.Series(values).astype('string').str.strip().str.lower()
    return s.isin(['true', '1', 't']).to_numpy()

def read_obs_column(group, name):

    node = group[name]
    if isinstance(node, h5py.Dataset):
        return node.asstr()[:] if node.dtype.kind in ('O', 'S', 'U') else node[:]

    cat = node['categories']
    categories = cat.asstr()[:] if cat.dtype.kind in ('O', 'S', 'U') else cat[:]
    codes = node['codes'][:]
    result = np.full(len(codes), None, dtype=object)
    valid = codes >= 0
    result[valid] = categories[codes[valid]]
    return result

def load_sep_lookup(path):
    table = pd.read_csv(path, usecols=['SpotID', 'sample_id', 'SampleID', 'SEP'], dtype='string')

    names = table[['sample_id', 'SampleID']].drop_duplicates()

    lookup = {}
    for sid, sub in table.groupby('sample_id', sort=False):
        sub = sub.copy()
        sub['hex_key'] = canonical_hex_ids(sub['SpotID'].reset_index(drop=True), sid).to_numpy()
        sub['SEP'] = sub['SEP'].str.strip().str.upper().str.replace('_', '-', regex=False)
        valid = sub.dropna(subset=['hex_key', 'SEP'])

        lookup[sid] = valid.drop_duplicates('hex_key').set_index('hex_key')['SEP']
    return lookup, names.set_index('sample_id')['SampleID'].to_dict()

def load_cell_annotations(path, sample_id, sep_lookup, display_name, swap_pt4_ab=True):

    with h5py.File(path, 'r') as f:
        obs = f['obs']
        fields = ['CellType', 'Clone', 'matched_hex15_id', 'hex15_match_accepted', 'hex36_match_accepted']
        d = pd.DataFrame({name: read_obs_column(obs, name) for name in fields})

        xy = f['obsm']['spatial'][:, :2]

        d[['x', 'y']] = xy

        image_shape = f['uns']['staining_image'].shape
        width, height = image_shape[:2]

        pixels_per_um = None
        if 'spatial_um' in f['obsm']:
            xy_um = f['obsm']['spatial_um'][:, :2]
            ranges = np.ptp(xy_um, axis=0)
            if np.isfinite(xy_um).all() and (ranges > 0).all():
                scales = np.ptp(xy, axis=0) / ranges

                pixels_per_um = float(scales[0])

    d['Clone_raw'] = clean_labels(d['Clone']).to_numpy()
    d['Clone_plot'] = d['Clone_raw'].copy()

    if swap_pt4_ab and display_name.split('-')[0] == 'Pt4':
        d['Clone_plot'] = d['Clone_plot'].replace({'A': 'B', 'B': 'A'})

    d['accepted15'] = accepted_flag(d['hex15_match_accepted'])
    d['accepted36'] = accepted_flag(d['hex36_match_accepted'])
    d['hex15_key'] = canonical_hex_ids(d['matched_hex15_id'], sample_id).to_numpy()
    d['SEP_matched'] = d['hex15_key'].map(sep_lookup[sample_id])
    d['MSEP_plot'] = d['SEP_matched'].where(d['accepted15'] & d['SEP_matched'].isin(MSEP_COLORS))
    d['eligible_clone'] = d['CellType'].eq('Epithelial') & d['accepted36'] & d['Clone_plot'].notna()

    d['eligible_msep'] = d['eligible_clone'] & d['MSEP_plot'].notna()
    info = {'sample_id': sample_id, 'SampleID': display_name, 'path': Path(path),
            'width': width, 'height': height, 'pixels_per_um': pixels_per_um}
    qc = {'sample_id': sample_id, 'SampleID': display_name,
          'N_cells': len(d), 'N_epithelial': int(d['CellType'].eq('Epithelial').sum()),
          'N_clone_colored': int(d['eligible_clone'].sum()),
          'N_msep_colored': int(d['eligible_msep'].sum()),
          'N_clone_without_valid_MSEP': int((d['eligible_clone'] & ~d['eligible_msep']).sum()),
          'N_clone_with_NSEP': int((d['eligible_clone'] & d['accepted15'] & d['SEP_matched'].str.startswith('N-SEP', na=False)).sum()),
          'N_clone_rejected_hex15': int((d['eligible_clone'] & ~d['accepted15']).sum())}
    return d, info, qc

def load_background(info, bounds=None, max_dim=1600, alpha=.18):
    width, height = info['width'], info['height']
    if bounds is None:
        bounds = (0, 0, width, height)
    xmin, ymin, xmax, ymax = map(int, bounds)

    step = max(1, int(np.ceil(max(xmax-xmin, ymax-ymin) / max_dim)))
    with h5py.File(info['path'], 'r') as f:
        img = f['uns']['staining_image']
        if img.ndim == 3:
            arr = img[xmin:xmax:step, ymin:ymax:step, :3].transpose(1, 0, 2)
        else:
            grey = img[xmin:xmax:step, ymin:ymax:step].T
            arr = np.repeat(grey[..., None], 3, axis=2)
        arr = arr.astype(np.float32)
        if arr.max() > 1:
            arr /= 255.
        mask = f['uns']['final_mask']
        if mask.shape == (height, width):
            region_mask = mask[ymin:ymax:step, xmin:xmax:step]
        elif mask.shape == (width, height):
            region_mask = mask[xmin:xmax:step, ymin:ymax:step].T

    rgba = np.concatenate([arr, (region_mask.astype(bool)*alpha)[..., None]], axis=-1)
    return rgba, (xmin, ymin, xmax, ymax)

def add_scale_bar(ax, bounds, pixels_per_um, requested_um):

    xmin, ymin, xmax, ymax = bounds
    w, h = xmax-xmin, ymax-ymin
    candidates = [v for v in [10,20,50,100,200,500,1000] if v <= requested_um and v*pixels_per_um <= .30*w]
    if not candidates:
        return
    length_um = max(candidates)
    x2, y = xmax-.05*w, ymax-.06*h
    ax.plot([x2-length_um*pixels_per_um, x2], [y,y], color='black', lw=1.6, solid_capstyle='butt')
    label = '1 mm' if length_um == 1000 else f'{length_um} μm'
    ax.text(x2-length_um*pixels_per_um/2, y-.025*h, label, ha='center', va='bottom', fontsize=8)

def draw_panel(ax, cells, field, palette, background, bounds, info, title, point_size=1.2, scale_um=1000, show_legend=True):
    xmin, ymin, xmax, ymax = bounds
    ax.imshow(background, extent=[xmin,xmax,ymax,ymin], interpolation='bilinear')
    if len(cells):
        colors = cells[field].map(palette)

        ax.scatter(cells['x'], cells['y'], c=colors.tolist(), s=point_size,
                   linewidths=0, rasterized=True)
    else:
        ax.text(.5,.5,'No eligible cells',transform=ax.transAxes,ha='center',va='center',fontsize=10)
    ax.set(xlim=(xmin,xmax), ylim=(ymax,ymin), aspect='equal', title=title)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)
    add_scale_bar(ax, bounds, info['pixels_per_um'], scale_um)
    present = set(cells[field].dropna())
    labels = [k for k in palette if k in present]
    if labels and show_legend:
        handles = [Line2D([],[],marker='o',linestyle='',markerfacecolor=palette[k],markeredgecolor='none',label=k,markersize=5) for k in labels]
        ax.legend(handles=handles,loc='upper center',bbox_to_anchor=(.5,-.025),
                  frameon=False,ncol=min(6,len(labels)),fontsize=8,handletextpad=.25,columnspacing=.7)

def save_figure(fig, prefix, dpi=300):
    prefix = Path(prefix)
    prefix.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(prefix)+'.pdf',bbox_inches='tight',facecolor='white',dpi=dpi)
    fig.savefig(str(prefix)+'.png',bbox_inches='tight',facecolor='white',dpi=dpi)


In [ ]:
ROOT = Path('/Volumes/B/Projects/2025Project/SpatialAnalysis/Project_ICC')
SAMPLE_DIR = ROOT / 'Data/ST/Samples'
SEP_CSV = ROOT / 'Data/ST/SEP/All_hex_MP_scores_for_h5ad.csv'
FIG_DIR = ROOT / 'Figure/ST/inferCNV/Cell_Clone_MSEP_ROI'

# ROI bounds are in image pixels: (xmin, ymin, xmax, ymax).
SAMPLE_CONFIGS = {

    'SR250530M3': {
        'crop': (3000, 3000, 6700, 5500),
        'target_clone': 'B',
        'point_size': 5,
        'figure_panel': 'Fig. 3n',
    },

    'SR250521M1': {
        'crop': (6700, 6200, 9600, 8100),
        'target_clone': 'A',
        'point_size': 10,
        'figure_panel': 'Fig. 3o',
    },
}
 
SWAP_PT4_AB = True
BACKGROUND_MAX_DIM = 1600
OVERVIEW_POINT_SIZE = 1.2
OVERVIEW_SCALE_UM = 1000
DPI = 400
SHOW_FIGURES = True


In [ ]:
def plot_overview(cells, info, config, output_dir):
    background, bounds = load_background(
        info, bounds=None, max_dim=BACKGROUND_MAX_DIM
    )
    xmin, ymin, xmax, ymax = map(int, config['crop'])

    clone_cells = cells.loc[cells['eligible_clone']]
    map_height = 7.0 * info['height'] / info['width']
    fig = plt.figure(figsize=(8.0, map_height + 1.6))
    try:
        grid = fig.add_gridspec(
            2, 1, height_ratios=[map_height, .7],
            left=.04, right=.97, top=.88, bottom=.06, hspace=.06
        )
        ax = fig.add_subplot(grid[0, 0])
        legend_ax = fig.add_subplot(grid[1, 0])
        draw_panel(
            ax, clone_cells, 'Clone_plot', CLONE_COLORS,
            background, bounds, info, 'Whole-section clone distribution',
            point_size=OVERVIEW_POINT_SIZE, scale_um=OVERVIEW_SCALE_UM,
            show_legend=False
        )

        roi_box = Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            fill=False, edgecolor='black', linewidth=1.4, zorder=5
        )
        ax.add_patch(roi_box)
        ax.annotate(
            f'ROI: Clone {config["target_clone"]}', xy=(xmin, ymin),
            xytext=(3, 3), textcoords='offset points',
            ha='left', va='bottom', fontsize=9, color='black',
            bbox=dict(facecolor='white', edgecolor='none', alpha=.8, pad=2),
            zorder=6
        )

        present = set(clone_cells['Clone_plot'].dropna())
        handles = [
            Line2D([], [], marker='o', linestyle='',
                   markerfacecolor=CLONE_COLORS[label], markeredgecolor='none',
                   label=f'Clone {label}', markersize=5)
            for label in CLONE_COLORS if label in present
        ]
        handles.append(Rectangle(
            (0, 0), 1, 1, fill=False, edgecolor='black', linewidth=1.2, label='ROI'
        ))
        legend_ax.axis('off')
        legend_ax.legend(
            handles=handles, loc='center', frameon=False,
            ncol=min(5, len(handles)), fontsize=9,
            handletextpad=.4, columnspacing=.9
        )
        panel_name = config.get('figure_panel', '')
        fig.suptitle(
            f'{panel_name} | {info["SampleID"]} | Whole-section overview',
            fontsize=13, y=.98
        )
        prefix = Path(output_dir) / (
            f'{info["SampleID"]}_{info["sample_id"]}_Clone_overview'
        )
        save_figure(fig, prefix, dpi=DPI)
        if SHOW_FIGURES:
            plt.show()
        return prefix
    finally:
        plt.close(fig)


In [ ]:
def plot_roi(cells, info, config, output_dir):
    target_clone = config['target_clone']

    background, bounds = load_background(
        info, bounds=config['crop'], max_dim=BACKGROUND_MAX_DIM
    )
    xmin, ymin, xmax, ymax = bounds
    inside = cells['x'].ge(xmin) & cells['x'].lt(xmax) & cells['y'].ge(ymin) & cells['y'].lt(ymax)

    clone_roi = cells.loc[inside & cells['eligible_clone']].copy()
    target_clone_roi = clone_roi.loc[clone_roi['Clone_plot'].eq(target_clone)].copy()

    other_clone_roi = clone_roi.loc[~clone_roi['Clone_plot'].eq(target_clone)].copy()
    other_clone_roi['Clone_focus'] = 'Other clones'
    target_clone_roi['Clone_focus'] = f'Clone {target_clone}'

    clone_context = pd.concat([other_clone_roi, target_clone_roi], ignore_index=True)
    clone_focus_colors = {
        'Other clones': OTHER_CLONE_COLOR,
        f'Clone {target_clone}': CLONE_COLORS[target_clone],
    }

    target_msep = target_clone_roi.loc[target_clone_roi['eligible_msep']].copy()

    counts = target_msep.groupby('MSEP_plot', observed=True).size().rename('N_cells').reset_index()
    msep_order = {name: i for i, name in enumerate(MSEP_COLORS)}
    counts['_order'] = counts['MSEP_plot'].map(msep_order)
    counts = counts.sort_values('_order').drop(columns='_order').reset_index(drop=True)
    counts['Fraction_among_valid_MSEP_cells'] = counts['N_cells'] / len(target_msep)
    counts.insert(0, 'Clone', target_clone)
    counts.insert(0, 'SampleID', info['SampleID'])

    aspect = (ymax - ymin) / (xmax - xmin)
    map_height = 5.0 * aspect
    legend_height = .75
    fig = plt.figure(figsize=(10.8, map_height + legend_height + 1.05))
    gs = fig.add_gridspec(
        2, 2, height_ratios=[map_height, legend_height],
        left=.035, right=.985, top=.86, bottom=.12, hspace=.07, wspace=.10
    )
    axes = [fig.add_subplot(gs[0, i]) for i in range(2)]
    legend_axes = [fig.add_subplot(gs[1, i]) for i in range(2)]
    for ax,lax,shown,field,palette,title in zip(axes,legend_axes,
        [clone_context, target_msep],
        ['Clone_focus', 'MSEP_plot'],
        [clone_focus_colors, MSEP_COLORS],
        [f'Clone {target_clone} in local tissue context',
         f'M-SEPs within Clone {target_clone}']):
        draw_panel(
            ax, shown, field, palette, background, bounds, info, title,
            point_size=config['point_size'], scale_um=50, show_legend=False
        )
        ax.set_anchor('S')
        present = set(shown[field].dropna())
        labels = [k for k in palette if k in present]
        lax.axis('off')
        if labels:
            handles = [Line2D([],[],marker='o',linestyle='',markerfacecolor=palette[k],
                             markeredgecolor='none',label=k,markersize=5) for k in labels]
            lax.legend(handles=handles,loc='upper center',frameon=False,ncol=min(5,len(labels)),
                       fontsize=8,handletextpad=.25,columnspacing=.65,borderaxespad=0)
    panel_name = config.get('figure_panel', '')
    fig.suptitle(
        f'{panel_name} | {info["SampleID"]} | Clone {target_clone}', fontsize=13, y=.985
    )
    fig.text(.025,.025,
        f'Left: {len(target_clone_roi):,} Clone {target_clone} cells highlighted among '
        f'{len(clone_roi):,} CNV-annotated epithelial cells. Right: {len(target_msep):,} '
        f'Clone {target_clone} cells with valid M-SEP labels.',
        fontsize=8,color='#555555',va='bottom')
    outdir = Path(output_dir)
    prefix = f'{info["SampleID"]}_{info["sample_id"]}_Clone_{target_clone}_MSEP_ROI'
    save_figure(fig, outdir / prefix, dpi=DPI)
    counts.to_csv(outdir / f'{prefix}_counts.csv', index=False)
    if SHOW_FIGURES: plt.show()
    plt.close(fig)
    return {
        'Figure_panel': panel_name, 'Target_clone': target_clone,
        'N_cells_in_ROI': int(inside.sum()),
        'N_clone_annotated_epithelial_ROI': len(clone_roi),
        'N_target_clone_ROI': len(target_clone_roi),
        'N_target_clone_with_MSEP_ROI': len(target_msep),
        'N_target_clone_without_valid_MSEP_ROI': len(target_clone_roi) - len(target_msep),
        'N_MSEP_categories_in_target_clone': target_msep['MSEP_plot'].nunique(),
        'xmin_px': xmin, 'ymin_px': ymin, 'xmax_px': xmax, 'ymax_px': ymax,
    }

def run_roi_examples():
    lookup,names=load_sep_lookup(SEP_CSV)
    outdir=Path(FIG_DIR)
    outdir.mkdir(parents=True,exist_ok=True)
    records=[]
    for sid,config in SAMPLE_CONFIGS.items():

        path=SAMPLE_DIR/sid/'integrated/cell_adata.h5ad'
        cells,info,qc=load_cell_annotations(path,sid,lookup,names[sid],SWAP_PT4_AB)
        plot_overview(cells, info, config, outdir)
        roi_qc=plot_roi(cells,info,config,outdir)
        records.append({**qc,**roi_qc})

        del cells
        gc.collect()
        pd.DataFrame(records).to_csv(outdir/'single_clone_MSEP_ROI_QC.csv',index=False)

    return pd.DataFrame(records)


In [ ]:
qc_roi = run_roi_examples()
qc_roi
